# AIRFAANS — Google Colab GPU runner

This notebook runs the AE 6394 AIRFAANS project on a single Colab GPU. It downloads the official AirfRANS dataset into the temporary Colab VM and writes checkpoints and result artifacts to Google Drive.

Before running: select **Runtime → Change runtime type → T4 GPU**. Free Colab runtimes are interruptible, so retain the Drive mount and use the resume cell after a disconnect.

In [ ]:
import os
import platform
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "GPU not detected. Select Runtime > Change runtime type > T4 GPU."
print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)

## Persistent output storage
Only checkpoints and compact result artifacts go to Drive. The 10 GB archive and extracted dataset stay on the temporary Colab disk to avoid filling Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_OUTPUT = "/content/drive/MyDrive/AIRFAANS/results"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print("Persistent results:", DRIVE_OUTPUT)

## Install AIRFAANS and dependencies
The repository is public, so no GitHub token is needed. Re-running this cell after a fresh Colab connection is safe.

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/AIRFAANS/.git ]; then
  git clone https://github.com/triasha72/AIRFAANS.git /content/AIRFAANS
else
  git -C /content/AIRFAANS pull --ff-only
fi
python -m pip install -q -e '/content/AIRFAANS[ml,airfrans,tracking,api]'
python -m pip install -q airfrans

## Download and verify the official dataset
This is approximately a 10 GB download and expands to approximately 14 GB. It must be repeated after Colab deletes the VM. The checked-in manifest identifies the expected archive SHA-256 as `6b301d75dee77fc6c7de6e551c44332be4acc84e30b253e9c60cfa756f6c96db`.

In [ ]:
from pathlib import Path

import airfrans as official_airfrans

DOWNLOAD_ROOT = Path("/content/airfrans_data")
case_files = list(DOWNLOAD_ROOT.rglob("*_internal.vtu")) if DOWNLOAD_ROOT.exists() else []
if len(case_files) != 1000:
    official_airfrans.dataset.download(root=str(DOWNLOAD_ROOT), unzip=True)
    case_files = list(DOWNLOAD_ROOT.rglob("*_internal.vtu"))
assert len(case_files) == 1000, f"Expected 1000 internal meshes, found {len(case_files)}"
DATASET_ROOT = str(case_files[0].parent.parent)
print("Dataset root:", DATASET_ROOT)
print("Verified simulations:", len(case_files))

## CUDA integration gate
Run this before spending hours on training. It uses real, disjoint AirfRANS cases but is deliberately bounded and is not a benchmark result.

In [ ]:
import subprocess

bounded_output = f"{DRIVE_OUTPUT}/bounded-colab-mlp-17"
command = [
    "airfaans",
    "train",
    "--dataset-root",
    DATASET_ROOT,
    "--manifest",
    "/content/AIRFAANS/data/manifests/airfrans_tasks_v0_1.json",
    "--config",
    "/content/AIRFAANS/configs/experiment_v0_1.yaml",
    "--model",
    "pointwise_mlp",
    "--task",
    "interpolation",
    "--seed",
    "17",
    "--output-dir",
    bounded_output,
    "--max-train-cases",
    "2",
    "--max-validation-cases",
    "1",
    "--max-test-cases",
    "1",
    "--epochs",
    "3",
    "--nodes-per-case",
    "256",
]
subprocess.run(command, check=True)

## First full treatment
Start with the pointwise MLP, interpolation task, seed 17. Do not run the complete 36-treatment matrix until this finishes and the artifact has been inspected. The command has no bounded-case overrides, so it uses the official split and performs complete-mesh test evaluation.

In [ ]:
FULL_MODEL = "pointwise_mlp"
FULL_TASK = "interpolation"
FULL_SEED = 17
full_output = f"{DRIVE_OUTPUT}/{FULL_TASK}-{FULL_MODEL}-{FULL_SEED}"
full_command = [
    "airfaans",
    "train",
    "--dataset-root",
    DATASET_ROOT,
    "--manifest",
    "/content/AIRFAANS/data/manifests/airfrans_tasks_v0_1.json",
    "--config",
    "/content/AIRFAANS/configs/experiment_v0_1.yaml",
    "--model",
    FULL_MODEL,
    "--task",
    FULL_TASK,
    "--seed",
    str(FULL_SEED),
    "--output-dir",
    full_output,
]
print("Running:", " ".join(full_command))
subprocess.run(full_command, check=True)

## Resume after interruption
After reconnecting, rerun the GPU, Drive, installation, and dataset cells. Set the same model/task/seed and execute this cell. Resume fails closed if the expected checkpoint is missing.

In [ ]:
resume_command = full_command + ["--resume"]
print("Resuming:", " ".join(resume_command))
subprocess.run(resume_command, check=True)

## Inspect and download the result
Do not treat a run as complete unless `result.json` exists beside `best.pt`. Preserve both files.

In [ ]:
import json

result_path = Path(full_output) / "result.json"
assert result_path.exists(), f"Missing result: {result_path}"
result = json.loads(result_path.read_text())
print(
    json.dumps(
        {
            "evidence_label": result["evidence_label"],
            "device": result["device"],
            "best_validation_mean_relative_l2": result["best_validation_mean_relative_l2"],
            "test_mean": result["test"]["mean"],
            "mean_force_absolute_error": result["test"].get("mean_force_absolute_error"),
            "checkpoint": result["checkpoint"],
        },
        indent=2,
    )
)